# Task 2 — CatBoost tuning (v6)

Source audit: `datacard.txt` calls `vip_tier` the **current** VIP level, describes `engagement_level` as an engagement classification, and `loyalty_tier` as based on spending. Neither the data card nor competition description gives a creation timestamp, lag, or formula. All three are present in Train and Test; this establishes schema availability, not pre-label availability. The tier-included branch assumes those values are available at prediction time. The tier-excluded branch is a robustness check.

This notebook reads the saved `eda_original_v2` card, uses only labeled Train, and excludes the three documented random-noise columns. The 20% holdout is consulted only after development-fold selection. Because earlier notebooks already used this holdout, it is a **reused** check, not an independent final test. Each configuration is fitted with fold-local preprocessing. Search uses a stratified 40,000-row development subset for breadth; finalists use all development rows. Draws with depth above 6 and more than 300 iterations were excluded on a runtime rule before their scores were used for selection. One excluded draw has a single completed fold in the cache and is ignored. Scores are macro F1; no Test labels are used.

In [1]:
from pathlib import Path
import json, sys, hashlib, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterSampler
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT/'src'
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task2.models.decision_rules import restore_categorical_nulls
SEED = set_seed()
DATA = 'eda_original_v2'
BASE = 't2_cat_original_no_noise_v2'
card = json.loads((SRC/'task2'/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train = pd.read_parquet(SRC/card['splits']['train']['path'])
assert train.player_id.is_unique and train.segment.notna().all()
y = train.segment.astype(int)
raw = train.drop(columns=['player_id','segment'])
categorical = raw.select_dtypes(include=['object']).columns.tolist()
X = restore_categorical_nulls(raw, categorical)
dev_pos, hold_pos = train_test_split(np.arange(len(y)), test_size=.20, stratify=y, random_state=SEED)
X_dev, X_hold = X.iloc[dev_pos].reset_index(drop=True), X.iloc[hold_pos]
y_dev, y_hold = y.iloc[dev_pos].reset_index(drop=True), y.iloc[hold_pos]
base_card = json.loads((SRC/'task2'/'save'/f'{BASE}.json').read_text(encoding='utf-8'))
baseline = joblib.load(SRC/base_card['model_path'])
base_pre = baseline.named_steps['preprocess']
base_params = baseline.named_steps['classifier'].get_params()
assert base_params['random_seed'] == SEED
assert len(X_dev)==83200 and len(X_hold)==20800
TIERS = ['loyalty_tier','engagement_level','vip_tier']
for col in TIERS: assert col in X_dev and col in X_hold
print('Development:',len(X_dev),'reused holdout:',len(X_hold))
print('Tier null fractions:',raw[TIERS].isna().mean().to_dict())

Development: 83200 reused holdout: 20800
Tier null fractions: {'loyalty_tier': 0.07716346153846154, 'engagement_level': 0.07848076923076923, 'vip_tier': 0.07645192307692307}


In [2]:
# CatBoost parameters are searched on development rows only.
# A modest, reproducible space covers tree capacity, shrinkage, regularization and sampling.
search_space = {
    'depth': [4,5,6,7,8],
    'iterations': [300,500,700],
    'learning_rate': [.025,.04,.06,.08],
    'l2_leaf_reg': [2,5,10,20],
    'random_strength': [0,1,3],
    'bagging_temperature': [0,.5,1],
}
draws = list(ParameterSampler(search_space, n_iter=12, random_state=SEED))
configs = [{'label':'baseline','params':{}}, *[
    {'label':f'search_{i:02d}','params':d} for i,d in enumerate(draws,1)
    if d['depth'] <= 6 or d['iterations'] <= 300
]]
search_pos, _ = train_test_split(np.arange(len(y_dev)), train_size=40000,
                                 stratify=y_dev, random_state=SEED)
X_search, y_search = X_dev.iloc[search_pos].reset_index(drop=True), y_dev.iloc[search_pos].reset_index(drop=True)
search_folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(X_search,y_search))
final_folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_dev,y_dev))
OUT = SRC/'task2'/'models'/'results'
OUT.mkdir(parents=True, exist_ok=True)
SEARCH_PATH = OUT/'t2_cat_search_subset_v6.json'
FINAL_PATH = OUT/'t2_cat_finalists_v6.json'
signature = hashlib.sha256(np.asarray(dev_pos,dtype=np.int64).tobytes()+np.asarray(y_dev,dtype=np.int8).tobytes()).hexdigest()
key = {'data':DATA,'created_at':card['created_at'],'seed':SEED,'split':signature,
       'search_pos_hash':hashlib.sha256(search_pos.tobytes()).hexdigest(),
       'search_space':search_space,'draws':draws}

def make_model(params, exclude_tiers=False):
    pre = clone(base_pre)
    if exclude_tiers:
        # Preserve the baseline preprocessor recipe, but omit all three tier fields.
        from sklearn.compose import ColumnTransformer
        cat_imp, cat_cols = pre.transformers[0][1], list(pre.transformers[0][2])
        num_imp, num_cols = pre.transformers[1][1], list(pre.transformers[1][2])
        cat_cols = [c for c in cat_cols if c not in TIERS]
        num_cols = [c for c in num_cols if c not in TIERS]
        pre = ColumnTransformer([('categorical',clone(cat_imp),cat_cols),
                                 ('numeric',clone(num_imp),num_cols)],
                                 remainder='drop',sparse_threshold=0)
        n_cat = len(cat_cols)
    else:
        n_cat = len(pre.transformers[0][2])
    p = {**base_params, **params, 'cat_features':list(range(n_cat))}
    return Pipeline([('preprocess',pre),('classifier',CatBoostClassifier(**p))])

def load_or_init(path, phase):
    if path.exists():
        data=json.loads(path.read_text(encoding='utf-8'))
        assert data['key']==key and data['phase']==phase, 'Config changed: use a new result name'
        return data
    return {'key':key,'phase':phase,'rows':[]}

WRITE_LOCK=threading.Lock()
def run_fold(record, path, label, params, fold_number, fit_pos, check_pos, exclude_tiers=False):
    if any(r['label']==label and r['fold']==fold_number for r in record['rows']): return
    start=time.monotonic()
    model=make_model(params,exclude_tiers=exclude_tiers)
    xx, yy = (X_search,y_search) if path==SEARCH_PATH else (X_dev,y_dev)
    model.fit(xx.iloc[fit_pos],yy.iloc[fit_pos])
    pred=np.asarray(model.predict(xx.iloc[check_pos])).reshape(-1).astype(int)
    score=f1_score(yy.iloc[check_pos],pred,average='macro')
    with WRITE_LOCK:
        record['rows'].append({'label':label,'fold':fold_number,'score':float(score),
                               'seconds':round(time.monotonic()-start,1)})
        path.write_text(json.dumps(record,indent=2)+'\n',encoding='utf-8')
    print(label,fold_number,round(score,5),'seconds',round(time.monotonic()-start,1),flush=True)

def run_config(record,path,label,params,folds,exclude_tiers=False):
    pending=[(i,fit,check) for i,(fit,check) in enumerate(folds,1)
             if not any(r['label']==label and r['fold']==i for r in record['rows'])]
    with ThreadPoolExecutor(max_workers=3) as pool:
        jobs=[pool.submit(run_fold,record,path,label,params,i,fit,check,exclude_tiers)
              for i,fit,check in pending]
        for job in as_completed(jobs): job.result()

In [3]:
# Stage 1: 9 runtime-screened sampled configs + baseline on three folds of a 40,000-row development subset.
search = load_or_init(SEARCH_PATH,'3-fold search')
for cfg in configs:
    run_config(search,SEARCH_PATH,cfg['label'],cfg['params'],search_folds)
active_labels={c['label'] for c in configs}
search_table=(pd.DataFrame(search['rows']).query('label in @active_labels').groupby('label').score
              .agg(['mean','std','count']).sort_values('mean',ascending=False))
assert search_table['count'].eq(3).all()
display(search_table.round(5))
top_labels=search_table.drop(index='baseline').head(3).index.tolist()
print('Prespecified finalist rule: top three search configs + baseline:',top_labels)

search_04 2 0.74559 seconds 101.1


search_04 3 0.75224 seconds 101.2


search_05 2 0.74922 seconds 97.2


search_05 3 0.75296 seconds 97.3


search_05

 1 0.7416 seconds 97.4


... (13 outputs left out)

search_12 2 0.74797 seconds 134.3


search_12 3 0.75619 seconds 134.5


,mean,std,count
label,,,
search_12,0.74826,0.00779,3
search_05,0.74793,0.00579,3
search_01,0.74783,0.00731,3
search_11,0.74774,0.00695,3
search_06,0.74736,0.00849,3
baseline,0.74715,0.00755,3
search_04,0.74530,0.00709,3
search_03,0.74380,0.00928,3
search_08,0.74282,0.00697,3


Prespecified finalist rule: top three search configs + baseline: ['search_12', 'search_05', 'search_01']


In [4]:
# Stage 2: matched five-fold checks on full development rows.
finalists=load_or_init(FINAL_PATH,'5-fold finalists')
by_label={c['label']:c['params'] for c in configs}
for label in ['baseline',*top_labels]:
    run_config(finalists,FINAL_PATH,label,by_label[label],final_folds)
# Also check the baseline recipe with all three temporally uncertain tiers removed.
run_config(finalists,FINAL_PATH,'no_tiers',{},final_folds,exclude_tiers=True)
final_table=(pd.DataFrame(finalists['rows']).groupby('label').score
             .agg(['mean','std','count']).sort_values('mean',ascending=False))
assert final_table['count'].eq(5).all()
display(final_table.round(5))
pivot=pd.DataFrame(finalists['rows']).pivot(index='fold',columns='label',values='score')
display(pivot.round(5))
winner=final_table.drop(index='no_tiers').index[0]
print('Development-fold winner:',winner,'vs baseline:',float(final_table.loc[winner,'mean']-final_table.loc['baseline','mean']))

baseline 1 0.75612 seconds 209.9


baseline 2 0.74606 seconds 210.3


baseline 3 0.75086 seconds 210.3


baseline 4 0.75805 seconds 161.1


baseline 5 0.75173 seconds 161.2


search_12 3 0.75277 seconds 289.4


... (18 outputs left out)

no_tiers 4 0.58561 seconds 156.5


,mean,std,count
label,,,
search_01,0.75409,0.00352,5
search_12,0.75336,0.00422,5
search_05,0.75301,0.00367,5
baseline,0.75256,0.00471,5
no_tiers,0.58736,0.00214,5


label,baseline,no_tiers,search_01,search_05,search_12
fold,,,,,
1,0.75612,0.59069,0.75818,0.75525,0.75638
2,0.74606,0.58753,0.74922,0.74760,0.74721
3,0.75086,0.58759,0.75364,0.75225,0.75277
4,0.75805,0.58561,0.75673,0.75735,0.75814
5,0.75173,0.58535,0.75271,0.75259,0.75229


Development-fold winner: search_01 vs baseline: 0.00153062701578921


In [5]:
# Fit, save, and evaluate the development-fold winner in one cell.
NAME='t2_cat_tuned_cv5_v6'
SAVE=SRC/'task2'/'save'
paths=[SAVE/f'{NAME}.joblib', SAVE/f'{NAME}.json', SAVE/f'{NAME}-evaluation.json']
if not any(p.exists() for p in paths):
    selected=make_model(by_label[winner])
    selected.fit(X_dev,y_dev)
    pred=np.asarray(selected.predict(X_hold)).reshape(-1).astype(int)
    save_model(task=2,name=NAME,model=selected,data_name=DATA,
        card={'idea':'CatBoost development-fold hyperparameter search',
              'selection':winner,'params':by_label[winner],
              'search':'12 sampled configs, runtime-screened to 9 plus baseline, on 3 folds of a stratified 40000-row development subset; top three plus baseline on 5 full development folds',
              'mean_cv_macro_F1':float(final_table.loc[winner,'mean']),
              'tier_timing':'unverified; this model includes all three tiers',
              'seed':SEED})
    result=evaluate_model(task=2,name=NAME,y_true=y_hold,y_pred=pred,
        card={'split':'reused stratified 20% holdout','seed':SEED,
              'selection':'development-fold winner; holdout not used in tuning'})
else:
    assert all(p.exists() for p in paths), 'Inspect incomplete saved artifact'
    saved=json.loads(paths[1].read_text(encoding='utf-8'))
    assert saved['data_name']==DATA and saved['card']['params']==by_label[winner]
    selected=joblib.load(paths[0])
    pred=np.asarray(selected.predict(X_hold)).reshape(-1).astype(int)
    result=json.loads(paths[2].read_text(encoding='utf-8'))
base_pred=np.asarray(baseline.predict(X_hold)).reshape(-1).astype(int)
print('Reused holdout baseline:',f1_score(y_hold,base_pred,average='macro'))
print('Reused holdout tuned:',result['score'])
print('Changed predictions:',int((base_pred!=pred).sum()))
print(classification_report(y_hold,pred,digits=5))

Reused holdout baseline: 0.7504144211991189
Reused holdout tuned: 0.7510954107114675
Changed predictions: 498
              precision    recall  f1-score   support

           0    0.78410   0.85618   0.81856      8191
           1    0.74697   0.74553   0.74625      5207
           2    0.74265   0.67309   0.70616      4206
           3    0.77836   0.69337   0.73341      3196

    accuracy                        0.76644     20800
   macro avg    0.76302   0.74204   0.75110     20800
weighted avg    0.76554   0.76644   0.76465     20800

